# 02b - Preprocessing of the full ESM25 dataset

Applies to every ESM25 record the preprocessing chosen on the development subset in notebook 02 (window, anchor,
resampling, spectrogram and scalogram on the common grid; parameters in the `preprocessing` section of
`configs/default.yaml`). No exploratory analysis is repeated here.

- The traces are read from the full SeisBench datasets in the cache, whatever `waveforms.source` says (so notebooks 01
  and 02 can stay on the development subset).
- Output: `data/processed/seisbench/features.h5` and `records.csv`; to train on it, set
  `training.feature_source: seisbench`.
- The run is **resumable**: if it stops (kernel restart, sleep, error), run the notebook again and it continues from
  the records not yet processed. If the file is already complete, the preprocessing is skipped.
- At the end: checks on the stored records and comparison with the development subset (the same records must give the
  same features).

## 0. Setup

In [ ]:
from dataclasses import replace
from pathlib import Path
import shutil

import numpy as np
import pandas as pd

from wqu.config import load_config
from wqu.data.exploration import numeric_summary_by_label
from wqu.data.loading import CV_COMPONENTS, load_esm25_metadata
from wqu.data.waveforms import iter_waveform_batches, stored_sampling_rate_hz
from wqu.preprocessing.pipeline import (
    FEATURE_INDEX_COLUMN,
    REPRESENTATIONS,
    feature_file_path,
    grid_from_config,
    preprocess_and_store,
    read_feature_records,
    read_features,
)

CONFIG_PATH = Path("..") / "configs" / "default.yaml"
FULL_SOURCE = "seisbench"
DEV_SOURCE = "dev_subset"
BYTES_PER_VALUE = 2
COMPARISON_BLOCK = 250

config = load_config(CONFIG_PATH)
config = replace(config, waveforms=replace(config.waveforms, source=FULL_SOURCE))
columns = config.columns
preprocessing = config.preprocessing
label_names = {subset.label: subset.name for subset in config.data.subsets}
grid = grid_from_config(preprocessing)

_, metadata = load_esm25_metadata(config)
sampling_rate_hz = stored_sampling_rate_hz(metadata)
features_path = feature_file_path(preprocessing, FULL_SOURCE)
print(f"Records: {len(metadata):,}; window {preprocessing.pre_anchor_s:g} s before to {preprocessing.post_anchor_s:g} s "
      f"after the anchor ({preprocessing.anchor_method}); grid {len(grid.frequencies_hz)} x {len(grid.times_s)}")
print(f"Output: {features_path}")

## 1. Disk space

Size of the feature file (both representations, float16) against the free space on the disk.

In [ ]:
values_per_record = len(REPRESENTATIONS) * len(CV_COMPONENTS) * len(grid.frequencies_hz) * len(grid.times_s)
required_bytes = len(metadata) * values_per_record * BYTES_PER_VALUE
features_path.parent.mkdir(parents=True, exist_ok=True)
free_bytes = shutil.disk_usage(features_path.parent).free
print(f"Feature file: {required_bytes / 1e9:.1f} GB; free space: {free_bytes / 1e9:.1f} GB")
if not features_path.exists() and free_bytes < required_bytes:
    raise RuntimeError("Not enough free disk space for the feature file.")

## 2. Preprocessing (resumable)

About two hours on a laptop. The progress bar starts from the records already processed in a previous run.

In [ ]:
if features_path.exists():
    stored_records = read_feature_records(features_path, columns.text)
    print(f"Feature file already complete: {len(stored_records):,} records")
else:
    stored_records = preprocess_and_store(
        metadata, lambda selected: iter_waveform_batches(selected, config), sampling_rate_hz, preprocessing,
        config.waveforms.zero_tolerance, features_path,
    )
    print(f"Stored {len(stored_records):,} records in {features_path}")

## 3. Checks on the stored records

Records per class, window statistics by class (to compare with notebook 02 on the development subset) and share of
records with particular windows.

In [ ]:
print(stored_records[columns.label].map(label_names).value_counts().to_string())
WINDOW_COLUMNS = ["anchor_time_s", "recorded_fraction_kept", "energy_fraction_kept",
                  *[f"missing_in_window_s_{component}" for component in CV_COMPONENTS]]
numeric_summary_by_label(stored_records, WINDOW_COLUMNS, columns.label).rename(index=label_names, level=columns.label).round(3)

In [ ]:
window_samples = stored_records["window_stop"] - stored_records["window_start"]
missing = stored_records[[f"missing_in_window_s_{component}" for component in CV_COMPONENTS]]
window_flags = pd.DataFrame(
    {
        "window starts before the recording": stored_records["window_start"] < 0,
        "some data missing in the window": missing.max(axis=1) > 0,
        "a component without data in the window": missing.max(axis=1) >= preprocessing.window_length_s,
        "energy kept < 95%": stored_records["energy_fraction_kept"] < 0.95,
    }
)
window_flags.groupby(stored_records[columns.label]).mean().T.rename(columns=label_names).round(3)

## 4. Comparison with the development subset

The records of the development subset are also in the full dataset: their windows and features must be the same in
the two files (the traces come from the same SeisBench datasets).

In [ ]:
dev_path = feature_file_path(preprocessing, DEV_SOURCE)
dev_records = read_feature_records(dev_path, columns.text)
full_rows = stored_records.loc[dev_records.index, FEATURE_INDEX_COLUMN].to_numpy()
dev_rows = dev_records[FEATURE_INDEX_COLUMN].to_numpy()
same_window = (stored_records.loc[dev_records.index, "window_start"].to_numpy() == dev_records["window_start"].to_numpy())
print(f"Development records: {len(dev_records):,}; same window start: {same_window.mean():.4f}")

differences = {}
for representation in REPRESENTATIONS:
    largest = 0.0
    for start in range(0, len(dev_rows), COMPARISON_BLOCK):
        stop = start + COMPARISON_BLOCK
        dev_values, _ = read_features(dev_path, representation, dev_rows[start:stop])
        full_values, _ = read_features(features_path, representation, full_rows[start:stop])
        largest = max(largest, float(np.abs(dev_values - full_values).max()))
    differences[representation] = largest
pd.Series(differences, name="largest difference (dB)")